## Introduction to Pinecone Indexes

![alt text](<Screenshot 2026-09-09 094638.png>)

### Pinecone
- Indexes
    - Stores vectors
    - Serve queries
    - Index contains records for each vector, including metadata
- Types:
    - Pod-Based:
        - Choose hardware to create the index pods
        - Pod type determines storage, query, latency, query throughout
    - Serverless:
        - No resources management
        - Indexes scale automatically
        - Run on cloud & store in blob
        - Easy to use & cheaper

In [1]:
from pinecone import Pinecone, ServerlessSpec
import os
from pathlib import Path
from dotenv import load_dotenv  # Import the specific functions from dotenv

# Find the workspace root by walking upward until the repository .env file is found.
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / ".env").exists():
    repo_root = repo_root.parent

env_path = repo_root / ".env"
load_dotenv(env_path, override=True)  # Always override any blank inherited variable.

# Retrieve the key
api_key = os.getenv("PINECONE_API_KEY")

# Safety check to make sure it loaded
if not api_key:
    raise ValueError("API Key is still missing! Double-check the variable name inside your .env file.")

# Initialize the client
pc = Pinecone(api_key=api_key)
print("Connected successfully! Pinecone client is ready.")

Connected successfully! Pinecone client is ready.


### Getting Started

In [ ]:
# create an index with the specified name, dimension, and serverless specification
pc.create_index(
    name="datacamp-index",
    dimension=1536,
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1",
    ),
)

# to add param afterwrds, you can use the following code snippet:
# metric = "dotproduct" or "cosine" or "euclidean"

### Managing Indexes

In [3]:
index = pc.Index("datacamp-index")
print(index.describe_index_stats())
print(pc.list_indexes())

DescribeIndexStatsResponse(dimension=1536, total_vector_count=100, metric='cosine', namespaces=1)
IndexList([<name='datacamp-index', ready=True>])


In [ ]:
# deleting index
pc.delete_index("index_name") # repalce with index name you want to delete

### Namespaces
- Containers for partitioning Indexes
    - Separate datasets
    - Data versioning
    - Separate groups

![alt text](<Screenshot 2026-09-09 103440.png>)

### Vector Ingestion

In [7]:
import random

# sample vectors to insert into the index
vectors_sample_1 = [
    {
        "id": "0",
        "values":[0.1, 0.2, 0.3],

    },
    {
        "id": "1",
        "values":[0.4, 0.5, 0.6],
    },

]

# other sample data
vectors_sample_2 = [
    {
        "id": "2",
        "values": [random.uniform(-1, 1) for _ in range(1536)], # Generates 1536 floats
        "metadata": {"genre": "comedy", "year": 2020},
    },
    {
        "id": "3",
        "values": [random.uniform(-1, 1) for _ in range(1536)], # Generates 1536 floats
        "metadata": {"genre": "drama", "year": 2021},
    }
]


In [5]:
# to check vectors dimension
vector_dims = [len(vector["values"]) == 1536 for vector in vectors_sample_2]
all(vector_dims)  # This will return True if all vectors have the correct dimension, False otherwise.

False

### Upserting Vectors
- .upsert(): Update or Insert

In [8]:
index.upsert(
    vectors=vectors_sample_2
)

UpsertResponse(upserted_count=2)

## Retrieving Vectors

- Accessing vectors
    - Fetching: Retrieve vectors based on their IDs
        - Read units: measure resources consumed during read operations (fetching, querying, listing)
    - Querying: Retrieve similar vectors to a input vector


### Fetching

In [ ]:
index.fetch(

    ids=["2", "3"],
    namespace="test-namespace"

)

FetchResponse(vectors={'2': Vector(id='2', values=[-0.374185205, -0.457807481, 0.195336565, ...1533 more], sparse_values=None, metadata={'genre': 'comedy', 'year': 2004}), '3': Vector(id='3', values=[0.912583768, -0.24664712, -0.321518689, ...1533 more], sparse_values=None, metadata={'genre': 'action', 'year': 2021})}, namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Thu, 10 Sep 2026 07:29:46 GMT', 'content-type': 'application/json', 'content-length': '38246', 'connection': 'keep-alive', 'x-pinecone-request-latency-ms': '302', 'x-envoy-upstream-service-time': '302', 'x-pinecone-response-duration-ms': '304', 'grpc-status': '0', 'server': 'envoy'}))

### Querying 

![alt text](<Screenshot 2026-09-10 153843.png>)

In [10]:
index.query(
    vector=[random.uniform(-1, 1) for _ in range(1536)],  # Random query vector
    top_k=2,
    include_metadata=True,
)

QueryResponse(matches=[ScoredVector(id='86', score=0.0689183325, values=[], metadata={'genre': 'comedy', 'year': 2018}), ScoredVector(id='62', score=0.0497803278, values=[], metadata={'genre': 'thriller', 'year': 2019})], namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Thu, 10 Sep 2026 07:40:05 GMT', 'content-type': 'application/json', 'content-length': '241', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '2', 'x-pinecone-request-latency-ms': '542', 'x-envoy-upstream-service-time': '44', 'x-pinecone-response-duration-ms': '543', 'grpc-status': '0', 'server': 'envoy'}))

**Metadata Filtering**

In [11]:
index.query(
    vector=[random.uniform(-1, 1) for _ in range(1536)],  # Random query vector
    top_k=2,
    filter={
        "genre": {"$eq": "comedy"},
        "year": 2020
        },  # Filter for genre "comedy"
    include_metadata=True,
)

QueryResponse(matches=[ScoredVector(id='2', score=-0.01359984, values=[], metadata={'genre': 'comedy', 'year': 2020}), ScoredVector(id='60', score=-0.0573527068, values=[], metadata={'genre': 'comedy', 'year': 2020})], namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Thu, 10 Sep 2026 08:04:47 GMT', 'content-type': 'application/json', 'content-length': '238', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '2', 'x-pinecone-request-latency-ms': '749', 'x-envoy-upstream-service-time': '259', 'x-pinecone-response-duration-ms': '750', 'grpc-status': '0', 'server': 'envoy'}))

**Updating & Deleting Vectors**

![alt text](<Screenshot 2026-09-10 161125.png>)

In [ ]:
index.update(
    id="2",
    values=[random.uniform(-1, 1) for _ in range(1536)],  # Update existing random vector values
    set_metadata={"genre": "comedy", "year": 2022}  # Updated metadata

)

# to delete vector and metadata, you can use the following code snippet:
index.delete(
    ids=["2", "3"],
    namespace="test-namespace"
)

## Batching Upserts

![alt text](<Screenshot 2026-09-10 163207.png>)


**Sequential Batching**

In [ ]:
import itertools

# create a chunk vector function
def chunks(iterable, batch_size=100):
    it = iter(iterable)
    chunk = tuple(itertools.islice(it, batch_size))
    while chunk:
        yield chunk
        chunk = tuple(itertools.islice(it, batch_size))

# solve rate & size limit but really slow
for chunk in chunks(vectors_sample_2, batch_size=1):
    index.upsert(vectors=chunk)

**Parallel Batching**
- plitting requests & sending them in parallel

In [ ]:
pc = Pinecone(api_key=api_key, pool_size=10)  # Adjust the pool size as needed
# pool_size or pool_threads for the parameter? 

with pc.Index("datacamp-index", pool_size=10) as index:
    async_result = [index.upsert(vectors=chunk, async_req=True)
                    for chunk in chunks(vectors_sample_2, batch_size=1)]
    [async_result.get() for async_result in async_result]


**Multitenancy & Namespaces**

![alt text](<Screenshot 2026-09-12 161208.png>)

- Serve multiple tenants in isolation
- Separate different customer's data
- Reduce query latency

**Multitenancy Strategies**
- Namespaces
- Metadata Filtering
- Separate Indexes

**Namespaces**

In [ ]:
# Namespaces

index.upsert(
    vectors=vectors_sample_2,
    namespace="test-namespace"
)

**Querying Vectors from Namespaces**

In [ ]:
query_result = index.query(
    vector=[random.uniform(-1, 1) for _ in range(1536)],  # Random query vector
    top_k=2,
    namespace="test-namespace",
    include_metadata=True,
)

### Semantic Search with Pinecone

![alt text](<Screenshot 2026-09-12 162711.png>)

**Semantic Search Engines**
- Embed & ingest documents into Pinecone Index
- Embed a user query
- Query the index with the embedded user query

In [ ]:
from openai import OpenAI

client = OpenAI(api_key="OPENAI_API_KEY")


**Ingesting Documents into Pinecone**

In [ ]:
import pandas as pd
import numpy as np
from uuid import uuid4

df = pd.read_csv("data/sample_data.csv")  # Replace with your actual CSV file path
batch_size = 100  # Adjust based on your needs

for batch in np.array_split(df, len(df) / batch_size):
    metadatas = [{"text_id":row["id"], "text":row["text"], "title":row["title"]} # add more if needed
                 for _, row in batch.iterrows()]
    texts = batch["text"].tolist()
    ids = [str(uuid4()) for _ in range(len(batch))]

    response = client.embeddings.create(
        input=texts,
        model="text-embedding-3-large"
    )

    embeds = [np.array(x.embedding) for x in response.data]
    index.updsert(
        vectors=zip(ids, embeds, metadatas),
        namespace="test-namespace"
    )
            

**Querying with Pinecone**

In [ ]:
query = ""
query_response = client.embeddings.create(
    input=query,
    model="text-embedding-3-large"
)

query_emb = query.response.data[0].embedding

retrieve_docs = index.query(
    vector=query_emb,
    top_k=5,
    namespace="test-namespace",
    include_metadata=True
)

for result in retrieve_docs["matches"]:
    print(f"{round(result['score'], 2)}: {result['metadata']['text']}")

### RAG Chatbot with Pinecone and OpenAI

![alt text](<Screenshot 2026-09-12 171741.png>)

- Embed User Query
- Retrieve Similar Documents
- Added Documents to Prompt

In [ ]:
def retrieve_documents(query, top_k, namespace, emb_model):
    query_response = client.embeddings.create(
        input=query,
        model=emb_model
    )

    query_emb = query_response.data[0].embedding

    retrieve_docs = []
    sources = []

    docs = index.query(
        vector=query_emb,
        top_k=top_k,
        namespace=namespace,
        include_metadata=True
    )

    for doc in docs["matches"]:
        retrieve_docs.append(doc["metadata"]["text"])
        sources.append((doc["metadata"]["title"], doc["metadata"]["text_id"]))

    return retrieve_docs, sources

def prompt_with_context_builder(query, docs):
    delim = "\n\n---\n\n"
    prompt_start = "Answer the following question based on the context provided. If the answer is not contained within the context, respond with 'I don't know.'\n\nContext:\n"
    prompt_end = f"\n\nQuestion: {query}"
    prompt = prompt_start + delim.join(docs) + prompt_end
    return prompt

def question_answering(prompt, sources, chat_model):
    sys_prompt = "You are a helpful assistant. Answer the question based on the provided context. If the answer is not contained within the context, respond with 'I don't know.'"
    res = client.chat.completions.create(
        model=chat_model,
        messages=[
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": prompt}     
        ], temperature=0.0
    )
    answer = res.choices[0].message.content.strip()
    answer += "\n\nSources:"
    for source in sources:
        answer += f"\n- {source[0]} (ID: {source[1]})"
    return answer

**Implementation**

In [ ]:
query = "Your query text here"  # Replace with your actual query text
documents = retrieve_documents(query, top_k=5, namespace="test-namespace", emb_model="text-embedding-3-large")
prompt_with_context = prompt_with_context_builder(query, documents[0])
answer = question_answering(prompt_with_context, sources, chat_model="gpt-4o")